# 🚀 Notebook do Professor (Demo) — Aula 10: Context Engineering para agentes

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 10/14 — Módulo 3: curadoria em loop agêntico · MCP overview**  
**⏱️ 1h40min**  
**🧠 Scratchpad · Context rot · MCP**  
**🔁 Andaime 55%**  

---

## 🎯 Objetivo da aula

Entender por que o contexto em agentes é um recurso ainda mais crítico que em chats simples — e dominar as estratégias de curadoria que mantêm a qualidade do agente em loops longos. Preparar o agente para a integração final da Aula 11.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz as soluções dos exercícios da aula.

---

# 🔬 Código da aula — slide a slide

### Slide 08 — Scratchpad — o contexto de trabalho do agente

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
# O que o AgentExecutor mantém internamente como scratchpad
scratchpad_turno_1 = """
Thought: Preciso buscar o prazo de garantia nos documentos.
Action: buscar_nos_documentos
Action Input: "prazo de garantia"
Observation: [manual.pdf, pág.15] O prazo de garantia é de 24 meses...
"""  # ~80 tokens

scratchpad_turno_2 = """
Thought: Encontrei 24 meses. Vou converter para dias.
Action: calcular
Action Input: "24 * 30"
Observation: 720
"""  # +40 tokens → total: ~120 tokens

# Após 10 iterações com resultados de busca ricos:
# scratchpad ≈ 10 × 800 chars ≈ ~2.000 tokens só de Observations
# + Thoughts (~500 tokens) + system + tools_schema (~1.000 tokens)
# Total: ~3.500–5.000 tokens por invocação do agente

# Inspecionar o scratchpad via intermediate_steps
resultado = executor.invoke({"input": "pergunta"})
for i, (action, obs) in enumerate(resultado["intermediate_steps"], 1):
    print(f"Iteração {i}: tool={action.tool}, obs_len={len(str(obs))} chars")
# → Iteração 1: tool=buscar_nos_documentos, obs_len=842 chars
# → Iteração 2: tool=calcular, obs_len=3 chars

### Slide 10 — Medir context rot em agente — qualidade vs. turnos

In [ ]:
import tiktoken, matplotlib.pyplot as plt

enc = tiktoken.encoding_for_model("gpt-4")

def contar_tokens_scratchpad(intermediate_steps) -> int:
    """Conta tokens acumulados no scratchpad."""
    texto = ""
    for action, obs in intermediate_steps:
        texto += f"{action.log}\n{obs}\n"
    return len(enc.encode(texto))

# Simular conversação longa — 15 perguntas em sequência
historico_scratchpad = []  # acumula manualmente para simular o contexto crescente
tokens_por_turno      = []
qualidade_por_turno   = []

for i, pergunta in enumerate(PERGUNTAS_TESTE, 1):
    resultado = executor.invoke({"input": pergunta})
    steps    = resultado["intermediate_steps"]
    tokens   = contar_tokens_scratchpad(steps)

    # Avaliar qualidade com LLM-as-judge (Aula 07)
    qualidade = faithfulness_manual(pergunta, resultado["output"],
                                     "\n".join(str(o) for _,o in steps))
    tokens_por_turno.append(tokens)
    qualidade_por_turno.append(qualidade)
    print(f"Turno {i:2d}: {tokens:5d} tokens · qualidade={qualidade:.2f}")

# Plotar context rot
fig, ax1 = plt.subplots(figsize=(9,4))
ax2 = ax1.twinx()
ax1.bar(range(len(tokens_por_turno)), tokens_por_turno, alpha=.4, color="steelblue", label="Tokens")
ax2.plot(qualidade_por_turno, color="#ED145B", marker="o", label="Qualidade")
plt.title("Context rot — tokens vs. qualidade por turno")
plt.show()

### Slide 13 — Compressão de Observations — resumir antes de inserir

In [ ]:
# Tool wrapper que comprime a Observation antes de retornar ao scratchpad
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm_mini = ChatOllama(model="gpt-oss:120b", temperature=0)

chain_resumir = (
    ChatPromptTemplate.from_template(
        "Resuma o texto abaixo em no máximo 3 frases, mantendo APENAS os fatos essenciais:\n\n{texto}"
    )
    | llm_mini | StrOutputParser()
)

@tool
def buscar_na_web_comprimida(query: str) -> str:
    """Use para buscar informações atuais na web. Retorna um resumo conciso dos resultados."""
    resultado_bruto = DuckDuckGoSearchRun().run(query)

    # Comprime ANTES de retornar ao scratchpad
    if len(resultado_bruto) > 500:
        return chain_resumir.invoke({"texto": resultado_bruto})
    return resultado_bruto  # já curto — não precisa resumir

# Diferença típica:
# Sem compressão: Observation = 800–1200 chars (~250 tokens)
# Com compressão: Observation = 150–250 chars (~55 tokens)
# Redução: ~78% dos tokens de Observation

### Slide 14 — Janela deslizante — custo fixo em loops longos

In [ ]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.messages import trim_messages

# Estratégia 1 — limitar via max_iterations (simples)
executor_limitado = AgentExecutor(
    agent=agente,
    tools=tools,
    max_iterations=5,       # mantém no máximo 5 iterações no scratchpad
    max_execution_time=60,  # para após 60 segundos (fallback de segurança)
    verbose=True,
)

# Estratégia 2 — resumo episódico entre turnos (avançado)
def resumir_scratchpad(intermediate_steps: list, max_steps: int = 3) -> list:
    """Mantém as últimas max_steps iterações e resume o resto."""
    if len(intermediate_steps) <= max_steps:
        return intermediate_steps

    antigas   = intermediate_steps[:-max_steps]
    recentes  = intermediate_steps[-max_steps:]

    # Resumir as iterações antigas em uma linha
    resumo = ", ".join(f"{a.tool}→{str(o)[:50]}" for a,o in antigas)
    print(f"[Resumo de {len(antigas)} iterações anteriores: {resumo}]")

    # Retorna só as recentes — scratchpad compacto
    return recentes

### Slide 15 — Memória episódica — persistir entre sessões

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from datetime import datetime

# Salvar resumo da sessão ao encerrar
def salvar_sessao(pergunta: str, resposta: str, session_id: str):
    """Persiste o par (pergunta, resposta) como memória episódica."""
    db_episodico.add_documents([Document(
        page_content=f"Pergunta: {pergunta}\nResposta: {resposta}",
        metadata={
            "session_id": session_id,
            "timestamp": datetime.now().isoformat(),
            "tipo": "episodio",
        },
    )])

# Recuperar contexto de sessões anteriores (memória episódica como tool)
@tool
def lembrar_sessoes_anteriores(query: str) -> str:
    """Use quando o usuário se referir a algo discutido em conversa anterior.
    Retorna resumos de interações passadas relevantes para a query."""
    docs = db_episodico.similarity_search(query, k=2,
                                          filter={"tipo": "episodio"})
    return "\n\n".join(d.page_content for d in docs)

### Slide 18 — MCP na prática — o seu agente como cliente MCP

In [ ]:
!pip install langchain-mcp-adapters -q  # adaptador oficial LangChain ↔ MCP

from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession
from mcp.client.stdio import stdio_client

# Conectar a um servidor MCP local (ex: servidor de filesystem)
async def conectar_mcp():
    async with stdio_client(
        {"command": "npx", "args": ["@modelcontextprotocol/server-filesystem", "/content"]}
    ) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            # Carrega tools do servidor MCP como tools LangChain
            mcp_tools = await load_mcp_tools(session)
            # → lista de ferramentas prontas para o AgentExecutor
            return mcp_tools

# Servidores MCP gratuitos disponíveis hoje:
# @modelcontextprotocol/server-filesystem  — ler/escrever arquivos
# @modelcontextprotocol/server-brave-search — busca web
# mcp-server-fetch                          — fazer requisições HTTP
# mcp-server-sqlite                         — consultar SQLite
# Catálogo: github.com/modelcontextprotocol/servers

### Slide 22 — Python novo desta aula

In [ ]:
# 1. ax.twinx() — segundo eixo Y no mesmo gráfico matplotlib
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()          # eixo direito — mesmo x, y diferente
ax1.bar(x, tokens, color="steelblue", alpha=.5)  # barras no eixo esquerdo
ax2.plot(qualidade, color="#ED145B", marker="o")  # linha no eixo direito

# 2. list slice negativo — últimas N iterações
steps   = ["a","b","c","d","e"]
recentes = steps[-3:]   # → ["c", "d", "e"] (últimas 3)
antigas  = steps[:-3]   # → ["a", "b"] (exceto as últimas 3)

# 3. datetime.now().isoformat() — timestamp para metadados
from datetime import datetime
ts = datetime.now().isoformat()  # → "2026-07-13T14:35:22.123456"

# 4. plt.bar com offset para barras lado a lado
x     = range(5)       # posições 0, 1, 2, 3, 4
width = 0.4
plt.bar([i-width/2 for i in x], vals_a, width, label="A")  # deslocado -0.2
plt.bar([i+width/2 for i in x], vals_b, width, label="B")  # deslocado +0.2

# 5. statistics.mean() para calcular redução média
import statistics
reducao_pct = (1 - statistics.mean(tok_com) / statistics.mean(tok_sem)) * 100
print(f"Redução média: {reducao_pct:.1f}%")

---

## 🏋️ Exercícios Resolvidos — versão professor (executar no Colab)

As quatro soluções prontas dos exercícios de fixação do notebook do aluno — rode em sala, uma a uma.


### Exercício 1 — Orçamento de contexto: medidor de scratchpad com tiktoken

A célula fecha o medidor: `encoding_for_model("gpt-4")`, a concatenação de `action.log` + `obs` e a contagem com `enc.encode`. Destacar a leitura típica: a iteração de busca web é a mais pesada (obs_len 800–1200 chars ≈ 200–300 tokens) — a candidata nº 1 à compressão.


In [ ]:
# ✅ Solução — Exercício 1 — medidor de scratchpad com tiktoken
!pip install -q langchain langchain-classic langchain-ollama langchain-community chromadb duckduckgo-search tiktoken

import tiktoken
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a10e1", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

prompt_react = hub.pull("hwchase17/react")

@tool
def buscar_nos_documentos(query: str) -> str:
    """Use quando o usuário perguntar sobre informações dos documentos do
    domínio (manuais, contratos, regulamentos). Retorna trechos relevantes
    com número de página. NÃO use para busca na web nem para cálculos."""
    docs = retriever.invoke(query)
    if not docs:
        return "Nenhum documento relevante encontrado."
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

@tool
def buscar_na_web(query: str) -> str:
    """Use quando precisar de informações atuais NÃO presentes nos documentos
    (cotações, notícias, eventos recentes). NÃO use para conteúdo interno
    do domínio do grupo."""
    return DuckDuckGoSearchRun().run(query)

@tool
def calcular(expressao: str) -> str:
    """Use para cálculos matemáticos. Recebe expressão Python válida
    (ex: '24*30'). NÃO use para buscar informações."""
    try:
        return str(eval(expressao, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"

tools = [buscar_nos_documentos, buscar_na_web, calcular]

executor_sem = AgentExecutor(
    agent=create_react_agent(llm, tools, prompt_react),
    tools=tools, verbose=False, max_iterations=5,
    handle_parsing_errors=True,
)

enc = tiktoken.encoding_for_model("gpt-4")

def contar_tokens_scratchpad(steps) -> int:
    """Soma os tokens de todos os Thoughts + Observations."""
    texto = ""
    for action, obs in steps:

        texto += f"{action.log}\n{obs}\n"
    return len(enc.encode(texto))

PERGUNTAS = [
    "Qual é a cláusula de garantia no documento?",
    "Qual é a cotação do dólar hoje?",
    "Qual o prazo de garantia em dias (meses × 30)?",
]
for q in PERGUNTAS:
    r = executor_sem.invoke({"input": q})
    print(f"\nPergunta: {q}")
    for i, (action, obs) in enumerate(r["intermediate_steps"], 1):
        acumulado = contar_tokens_scratchpad(r["intermediate_steps"][:i])
        print(f"  Iteração {i}: tool={action.tool:22s} "
              f"obs_len={len(str(obs)):4d} · tokens_acumulados={acumulado}")

# Leitura típica: a iteração de busca web é a mais pesada (obs_len 800–1200
# chars ≈ 200–300 tokens); a da calculadora tem obs_len de 2–5 chars.


### Exercício 2 — Compressão de Observations: medir o ganho

A célula fecha `chain_resumir` com `StrOutputParser()`, o limite de 500 chars e a chamada `chain_resumir.invoke` dentro da tool. Destacar a redução típica de ~70–80% nas buscas web — e a conferência resposta a resposta dos fatos essenciais (números, datas) que o resumo possa ter descartado.


In [ ]:
# ✅ Solução — Exercício 2 — ganho da compressão de Observations
!pip install -q langchain langchain-classic langchain-ollama langchain-community chromadb duckduckgo-search tiktoken

import tiktoken
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a10e2", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

prompt_react = hub.pull("hwchase17/react")

@tool
def buscar_nos_documentos(query: str) -> str:
    """Use quando o usuário perguntar sobre informações dos documentos do
    domínio (manuais, contratos, regulamentos). Retorna trechos relevantes
    com número de página. NÃO use para busca na web nem para cálculos."""
    docs = retriever.invoke(query)
    if not docs:
        return "Nenhum documento relevante encontrado."
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

@tool
def buscar_na_web(query: str) -> str:
    """Use quando precisar de informações atuais NÃO presentes nos documentos
    (cotações, notícias, eventos recentes). NÃO use para conteúdo interno
    do domínio do grupo."""
    return DuckDuckGoSearchRun().run(query)

@tool
def calcular(expressao: str) -> str:
    """Use para cálculos matemáticos. Recebe expressão Python válida
    (ex: '24*30'). NÃO use para buscar informações."""
    try:
        return str(eval(expressao, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"

tools = [buscar_nos_documentos, buscar_na_web, calcular]

import statistics
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

enc = tiktoken.encoding_for_model("gpt-4")

def contar_tokens_scratchpad(steps) -> int:
    """Soma os tokens de todos os Thoughts + Observations."""
    texto = ""
    for action, obs in steps:
        texto += f"{action.log}\n{obs}\n"
    return len(enc.encode(texto))

chain_resumir = (
    ChatPromptTemplate.from_template(
        "Resuma em no máximo 3 frases, preservando APENAS os fatos essenciais:\n\n{texto}")
    | llm | StrOutputParser()
)

@tool
def buscar_na_web_comprimida(query: str) -> str:
    """Use para buscar informações atuais na web. Retorna resumo conciso."""
    bruto = DuckDuckGoSearchRun().run(query)
    if len(bruto) > 500:
        return chain_resumir.invoke({"texto": bruto})
    return bruto

tools_sem = [buscar_nos_documentos, buscar_na_web, calcular]
tools_com = [buscar_nos_documentos, buscar_na_web_comprimida, calcular]
executor_sem = AgentExecutor(
    agent=create_react_agent(llm, tools_sem, prompt_react),
    tools=tools_sem, max_iterations=5, handle_parsing_errors=True)
executor_com = AgentExecutor(
    agent=create_react_agent(llm, tools_com, prompt_react),
    tools=tools_com, max_iterations=5, handle_parsing_errors=True)

PERGUNTAS = [
    "Qual é a cotação do dólar hoje?",
    "Quais são as últimas notícias sobre o tema do domínio?",
    "Qual o prazo de garantia em dias (meses × 30)?",
]
tok_sem, tok_com = [], []
for q in PERGUNTAS:
    tok_sem.append(contar_tokens_scratchpad(
        executor_sem.invoke({"input": q})["intermediate_steps"]))
    tok_com.append(contar_tokens_scratchpad(
        executor_com.invoke({"input": q})["intermediate_steps"]))

reducao = (1 - statistics.mean(tok_com) / statistics.mean(tok_sem)) * 100
print(f"Sem compressão : {tok_sem}")
print(f"Com compressão : {tok_com}")
print(f"Redução média  : {reducao:.1f}%")
# Típico: ~70–80% de redução nas buscas web — o ponto a conferir é se o
# resumo preservou os fatos essenciais (números, datas).


### Exercício 3 — MCP: seu agente como cliente MCP

A célula fecha o handshake (`session.initialize()`), o discovery (`load_mcp_tools(session)`) e o executor só com as tools MCP descobertas. Destacar a diferença arquitetural: com `@tool` quem executa é o processo Python e a description é sua; no MCP a tool roda no servidor externo, que publica name + description + schema.


In [ ]:
# ✅ Solução — Exercício 3 — agente como cliente MCP
!pip install -q langchain langchain-classic langchain-ollama langchain-community chromadb langchain-mcp-adapters

from langchain_ollama import ChatOllama
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm = ChatOllama(model="gpt-oss:120b", temperature=0)
prompt_react = hub.pull("hwchase17/react")

!node --version > /dev/null 2>&1 || apt-get install -y -qq nodejs npm > /dev/null 2>&1

from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession
from mcp.client.stdio import stdio_client

async def conectar_mcp():
    async with stdio_client(
        {"command": "npx",
         "args": ["@modelcontextprotocol/server-filesystem", "/content"]}
    ) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            mcp_tools = await load_mcp_tools(session)
            print(f"Tools MCP descobertas: {[t.name for t in mcp_tools]}")
            return mcp_tools

mcp_tools = await conectar_mcp()
executor_mcp = AgentExecutor(
    agent=create_react_agent(llm, mcp_tools, prompt_react),
    tools=mcp_tools,
    verbose=True, max_iterations=5,
)
print(executor_mcp.invoke(
    {"input": "Liste os arquivos em /content e mostre as 5 primeiras linhas do primeiro .txt."}
)["output"])
# Arquitetura: a tool roda no SERVIDOR MCP (processo externo) — o agente é
# apenas cliente; name/description/schema vêm do servidor, não do seu código.


### Exercício 4 — Memória episódica: persistir entre sessões

A célula fecha `add_documents`, o metadata `tipo="episodio"` + timestamp, o filtro do `similarity_search` e o roster com a 4ª tool. Destacar o ciclo de validação: ao perguntar sobre a sessão anterior, o agente aciona `lembrar_sessoes_anteriores` e cita o par (pergunta, resposta) salvo.


In [ ]:
# ✅ Solução — Exercício 4 — memória episódica entre sessões
!pip install -q langchain langchain-classic langchain-ollama langchain-community chromadb duckduckgo-search tiktoken

import tiktoken
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima do domínio — célula autocontida (substitui o /content/ckp02)
db = Chroma(collection_name="mini_a10e4", embedding_function=embeddings)
db.add_texts(
    [
        "O prazo de garantia do produto é de 24 meses a partir da data da compra.",
        "O suporte técnico é acionado pelo canal oficial, com resposta em até 48h úteis.",
        "A cobertura da garantia exclui mau uso e defeitos causados por terceiros.",
    ],
    metadatas=[{"source": "manual.pdf", "page": 13},
               {"source": "manual.pdf", "page": 20},
               {"source": "contrato.pdf", "page": 2}],
)
retriever = db.as_retriever(search_kwargs={"k": 3})

prompt_react = hub.pull("hwchase17/react")

@tool
def buscar_nos_documentos(query: str) -> str:
    """Use quando o usuário perguntar sobre informações dos documentos do
    domínio (manuais, contratos, regulamentos). Retorna trechos relevantes
    com número de página. NÃO use para busca na web nem para cálculos."""
    docs = retriever.invoke(query)
    if not docs:
        return "Nenhum documento relevante encontrado."
    return "\n\n".join(
        f"[{d.metadata.get('source', '?')}, pág.{d.metadata.get('page', 0) + 1}] {d.page_content}"
        for d in docs
    )

@tool
def buscar_na_web(query: str) -> str:
    """Use quando precisar de informações atuais NÃO presentes nos documentos
    (cotações, notícias, eventos recentes). NÃO use para conteúdo interno
    do domínio do grupo."""
    return DuckDuckGoSearchRun().run(query)

@tool
def calcular(expressao: str) -> str:
    """Use para cálculos matemáticos. Recebe expressão Python válida
    (ex: '24*30'). NÃO use para buscar informações."""
    try:
        return str(eval(expressao, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"

tools = [buscar_nos_documentos, buscar_na_web, calcular]

from datetime import datetime
from langchain_core.documents import Document

db_episodico = Chroma(persist_directory="/content/ckp03_episodico", embedding_function=embeddings)

def salvar_sessao(pergunta: str, resposta: str, session_id: str):
    db_episodico.add_documents([Document(
        page_content=f"Pergunta: {pergunta}\nResposta: {resposta}",
        metadata={"session_id": session_id, "tipo": "episodio",
                  "timestamp": datetime.now().isoformat()},
    )])

@tool
def lembrar_sessoes_anteriores(query: str) -> str:
    """Use quando o usuário se referir a algo discutido em conversa anterior.
    Retorna resumos de interações passadas relevantes para a query."""
    docs = db_episodico.similarity_search(query, k=2, filter={"tipo": "episodio"})
    return "\n\n".join(d.page_content for d in docs) if docs else "Nada encontrado."

tools4    = [buscar_nos_documentos, buscar_na_web, calcular, lembrar_sessoes_anteriores]
agente4   = create_react_agent(llm, tools4, prompt_react)
executor4 = AgentExecutor(agent=agente4, tools=tools4, verbose=True,
                          max_iterations=5, handle_parsing_errors=True)

# Ciclo de validação: salvar episódio → nova sessão → agente aciona a memória
salvar_sessao("Qual o prazo de garantia?", "24 meses (pág. 15)", "sessao-1")
print(executor4.invoke({"input": "O que eu perguntei na sessão anterior?"})["output"])


## 📚 Referências da aula

- Blog Anthropic Engineering — "Context Engineering for AI Agents" (setembro 2025). Fonte primária desta aula — princípio da ação mínima, tipos de memória, curadoria de scratchpad. anthropic.com/engineering/context-engineering
- Docs Model Context Protocol — Especificação oficial, servidores disponíveis e guia de implementação. modelcontextprotocol.io
- Paper Liu, N. et al. — "Lost in the Middle: How Language Models Use Long Contexts." EMNLP, 2023. A base empírica do context rot em contextos longos. arxiv.org/abs/2307.03172
- Docs LangChain MCP Adapters — Integrar servidores MCP como tools LangChain. github.com/langchain-ai/langchain-mcp-adapters
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 2 — Agentes e ambientes: a analogia memória=RAM / conhecimento=HD que fundamenta os 3 tipos de memória agêntica.
- Ebook Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 8: Memory Management — a distinção Short-Term vs. Long-Term por trás do scratchpad e da memória episódica desta aula.

---

**Próxima Aula — Aula 11 · 26/10** — Aula Integradora — Agente com RAG + Gradio ao vivo
  
100% lab. Integrar tudo. Publicar URL pública. Entregar CKP03.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*